# Retraining from scratch

We run these models separately from the main experiment so that, when a new unlearned model is made, these checkpoints can be quickly pulled, measured, and then set aside again.

### Imports

In [8]:
import sys
print(sys.version)

3.9.25 (main, Apr 17 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]


In [9]:
import os
import json

In [10]:
%ls

data/                              __pycache__/
evaluation/                        README.md
master_auditor.ipynb               results/
master_hyperparams.py              results_to_replicate.txt
master_pretraining.ipynb           trainer/
master_retrain_from_scratch.ipynb  unlearn/
master_unlearning.ipynb            visualize_pretraining_results.ipynb
models/                            visualize_results.ipynb
_old/                              wandb/


In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [12]:

from master_hyperparams import hyperparams
device = "cuda" if torch.cuda.is_available() else "cpu"


# ------- MAIN THINGS TO EDIT FOR THIS RUN ------- #
dataset = "CIFAR10"
model_class = "ResNet"
unlearning_type = "class"
# ------------------------------------------------- #

hp = hyperparams[dataset]
model_hp = hp[model_class]

retrain_config = {

    "description": "Retrain from Scratch - Resnet CIFAR10",
    
    "device": device,
    "model_class": model_class,
    "unlearning_type": unlearning_type,
    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": 1024,  # larger batch for faster pretraining
        "num_workers": hp["num_workers"],
        "item_to_unlearn": hp["items_to_unlearn"][unlearning_type]
        },

    "training": {
        "num_epochs": model_hp["training"]["num_epochs"],
        "num_runs": 3,
        "learning_rate": model_hp["training"]["learning_rate"],
        "weight_decay": model_hp["training"]["weight_decay"],
        "batch_print_freq": 12,
        },
}


### Protocol for several runs

In [13]:
import wandb
wandb.login()

True

In [14]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
import json
from data.utils import setup_seed
import time
from data.utils import split_forget_retain, split_random

def run_retrain_from_scratch(config, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING RETRAINING FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    setup_seed(config["GRAND_SEED"])

    # create a subfolder for saving model checkpoints for this retraining
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}", "retrain_from_scratch")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)


    # Save the config for this retraining to the main checkpoints folder
    with open(os.path.join(checkpoint_subfolder, "retrain_from_scratch_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # ... decide what we're unlearning
    item_to_unlearn = config["data"]["item_to_unlearn"]

    # ... announce what we're unlearning
    retrain_name = f"{config['data']['dataset']}_{config['model_class']}_{config['training']['num_epochs']}_epochs_{config['unlearning_type']}_{item_to_unlearn}"
    print("-"*15 + "    " + retrain_name + "\n")
    

    # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
    
    class_param = item_to_unlearn if config['unlearning_type'] == "class" else None
    percent_param = item_to_unlearn if config['unlearning_type'] == "percent" else None

    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #

    # ...  ------------- get some unlearning data for this config ------------------- #
    # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

    # only need `train`
    marked_train_loader, _, _ = load_dataloaders_for_experiment(
        name = config["data"]["dataset"],
        batch_size=config["data"]["batch_size"], 
        num_workers=config["data"]["num_workers"], 
        seed = config["GRAND_SEED"], 
        class_to_replace=class_param, 
        percent_to_replace=percent_param, 
        only_mark=True,
        val=False
        )
    # we make sure forget and retain sets are shuffled, to allow randomness across runs
    # only need `retain`
    print("Training - forget vs retain split:")
    _, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
    

    
    # ... and do a bunch of runs, where ...
    for i in range(1,  config["training"]["num_runs"]+1):

    
        # ----------------------------------------------------------------------------------- #
        # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
        # ----------------------------------------------------------------------------------- #

        run_seed = config["GRAND_SEED"] * 1000 + i
        setup_seed(run_seed)
        
            # ... open new wandb session per run
        wandb.init(
            project="Verifying-Unlearning-2026",
            name=f"{run_seed}_retrain_{retrain_name}",
            config=config,
            reinit= "finish_previous"
            )
        
        print(f" ----- Retraining from scratch for run {i}, {retrain_name} ----- \n")
            
        # ... init a fresh model, opt, criterion, and scheduler for this run_seed
        empty_model = init_model(model_class = config["model_class"], num_classes = config['data']["num_classes"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            opt, 
            T_max=config["training"]["num_epochs"], 
            eta_min=1e-6
            )

        # ... do the training
        retrain_name = f"retrain_run_{i}_{retrain_name}"
        retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
        start = time.time() # EVENTUALLY NEEDS TO BE MEASURED SOME OTHER WAY
        retrained_model, opt, scheduler, retrain_retain_loss, retrain_retain_acc, retrain_retain_entr, retrain_retain_m_entr = training_regimen_lr_annealing(
            empty_model, 
            retain_loader,
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = retrain_checkpoint_path,
            print_freq = config["training"]["batch_print_freq"],
            w_and_b = True
            )
        end = time.time()
        wandb.log({"run time efficiency": end - start})
        

        # closes retrain wandb session
        wandb.finish()


    print("-"*75)
    print("-"*19 + "  " + f'FINISHED RETRAIN FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*75 + "\n")
    

### Check metrics on unlearned models

In [15]:
# MAKE A RANDOM SEED
retrain_config["GRAND_SEED"] = 4
# DO EXP
run_retrain_from_scratch(config = retrain_config, checkpoint_folder="models/model_checkpoints")

===================  RUNNING RETRAINING FROM SCRATCH, SEED 4  ===================

setup random seed = 4
All models will be of class ResNet.

---------------    CIFAR10_ResNet_100_epochs_class_5

========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replaced class 5 in train
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain split:
Forget set: 5000 items
Retain set: 45000 items


setup random seed = 4001


 ----- Retraining from scratch for run 1, CIFAR10_ResNet_100_epochs_class_5 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.6736 (1.9864)	Accuracy 33.301 (27.010)	Entropy 1.6291 (1.7809)	M-Entropy 1.6051 (1.9207)	Time 6.33
Epoch: [1][23/44]	Loss 1.4630 (1.7637)	Accuracy 44.238 (34.578)	Entropy 1.4506 (1.6587)	M-Entropy 1.4083 (1.6935)	Time 2.52
Epoch: [1][35/44]	Loss 1.3003 (1.6366)	Accuracy 52.344 (39.293)	Entropy 1.3429 (1.5694)	M-Entropy 1.2500 (1.5716)	Time 2.53
train_accuracy (epoch) 41.780
Epoch 1 | LR: 1.0e-03 | RAM: 2.01GB | VRAM: 7.08GB | Weight Norm: 111.725
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.0698 (1.1703)	Accuracy 61.328 (56.982)	Entropy 1.1259 (1.1944)	M-Entropy 1.0409 (1.1442)	Time 3.23
Epoch: [2][23/44]	Loss 1.0593 (1.1293)	Accuracy 61.621 (58.720)	Entropy 1.0723 (1.1431)	M-Entropy 1.0462 (1.1128)	Time 2.52
Epoch: [2][35/44]	Loss 0.9806 (1.0893)	Accuracy 65.137 (60.395)	Entropy 0.9945 (1.1012)	M-Entropy 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁███████████████████████████████████████
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
learning_rate,█████████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
run time efficiency,▁
time (batch),▂▇▁▁▂▁█▂▂▇▂▂▂▂▂█▂█▆▁▁▇▂▆▁▁▁█▂▁▁▇▁▇▂▂▂▇█▂
train_acc (batch),▁▂▄▄▆▆▇▇▇▇▇▇▇▇▇█████████████████████████
train_acc (full),▁▃▅▅▅▆▆▆▆▇▇▇▇▇▇█████████████████████████
train_entropy (batch),█▇▇▆▆▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▇▆▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


setup random seed = 4002


...

setup random seed = 4003


 ----- Retraining from scratch for run 3, retrain_run_2_retrain_run_1_CIFAR10_ResNet_100_epochs_class_5 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.6514 (2.0095)	Accuracy 39.062 (27.954)	Entropy 1.6159 (1.7422)	M-Entropy 1.5637 (1.9694)	Time 3.18
Epoch: [1][23/44]	Loss 1.4594 (1.7682)	Accuracy 45.020 (35.307)	Entropy 1.4506 (1.6314)	M-Entropy 1.4004 (1.7105)	Time 2.45
Epoch: [1][35/44]	Loss 1.3648 (1.6414)	Accuracy 49.121 (39.621)	Entropy 1.2859 (1.5509)	M-Entropy 1.3666 (1.5853)	Time 2.56
train_accuracy (epoch) 42.107
Epoch 1 | LR: 1.0e-03 | RAM: 2.05GB | VRAM: 7.09GB | Weight Norm: 111.827
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.1479 (1.1963)	Accuracy 56.543 (56.291)	Entropy 1.1735 (1.2035)	M-Entropy 1.1218 (1.1745)	Time 3.27
Epoch: [2][23/44]	Loss 1.0837 (1.1484)	Accuracy 59.375 (58.146)	Entropy 1.0627 (1.1527)	M-Entropy 1.0917 (1.1338)	Time 2.46
Epoch: [2][35/44]	Loss 1.0053 (1.1180)	Accuracy 64.453 (59.239)	Entrop

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▇▇▇▇█████████████████████▁▆▆▇▇▇▇▇▇▇▇▇▇▇▇
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇████
learning_rate,███████▇▇▇▇▆▆▆▆▅▅▅▅▅▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
run time efficiency,▁
time (batch),▁▃▃▁▁▁▁▁▅▁▃▁▁▃▅▁▁▃▃▁▅▅▆▇▄█▆▁▃▁▁▃▃▁▁▃▁▁▃▁
train_acc (batch),▁▂▄▆▆▇▇▇▇▇▇▇▇▇▇█████████████████████████
train_acc (full),▁▂▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████████████
train_entropy (batch),█▇▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


---------------------------------------------------------------------------
-------------------  FINISHED RETRAIN FROM SCRATCH, SEED 4  -------------------
---------------------------------------------------------------------------

